## Data Loading - International Football Results 1872 to 2026

Loading in data using kaggle. Data used: https://www.kaggle.com/datasets/martj42/international-football-results-from-1872-to-2017/data

This dataset includes 49,393 results of international football matches starting from the very first official match in 1872 up to 2025. The matches range from FIFA World Cup to FIFI Wild Cup to regular friendly matches. The matches are strictly men's full internationals and the data does not include Olympic Games or matches where at least one of the teams was the nation's B-team, U-23 or a league select team.

In [167]:
import shutil
import kagglehub
import pandas as pd
import numpy as np
import seaborn as sns
import datetime as dt
from pathlib import Path

DATA_DIRECTORY = Path("../data/international-football-results")

def get_data_path(data_directory:Path) -> Path:

    if data_directory.exists():
        path = data_directory
        print("Using cached dataset at:", path)
    else:
        download_path = kagglehub.dataset_download("martj42/international-football-results-from-1872-to-2017")
        data_directory.parent.mkdir(parents=True, exist_ok=True)
        shutil.copytree(download_path, data_directory)
        path = data_directory
        print("Downloaded dataset to:", path)

    return path


def load_data(data_directory: Path) -> dict[str,pd.DataFrame]:
    frames = {file_path.stem : pd.read_csv(file_path) for file_path in sorted(data_directory.glob('*csv'))}

    for frame in frames.values():
        if 'date' in frame.columns:
            frame['date'] = pd.to_datetime(frame['date'],errors = 'coerce')
    
    return frames



frames = load_data(DATA_DIRECTORY)

results = frames['results']
shootouts = frames['shootouts']
goalscorers = frames['goalscorers']
former_names = frames['former_names']



In [168]:
def string_column(df,col,cardinality_threshold):
    unique_values = df[col].nunique()
    print(f'Total Unique Values in {col} = {unique_values}')
    if unique_values <= cardinality_threshold:
        print(df[col].value_counts())
    else:
        print('\n\nTop 10 Most Frequent Values')
        print(df[col].value_counts().head(10))
        print('\n\nBottom 10 Most Frequent Values')
        print(df[col].value_counts().tail(20))

def number_column(df,col):
    print(f'\n\n=== Summary Statistic {col}===')
    print(f' Max = {df[col].max():.2f}')
    print(f' Mean = {df[col].mean():.2f}')
    #print(f' Mode = {df[col].mode():.2f}')
    print(f' Median = {df[col].median():.2f}')
    print(f' Min = {df[col].min():.2f}')

def date_column(df,col):

    df = df.copy()

    print(f'\n\n=== Summary Statistic {col}===')

    print(f'Earliest Date == {df[col].min()}')
    print(f'Latest Date == {df[col].max()}')

    df['month'] = df[col].dt.month
    df['year'] = df[col].dt.year
    

    print(df[col].value_counts().head(25))

    print('\n\n',df['year'].value_counts().head(25))

    print('\n\n',df['month'].value_counts())

    #print(cross_count.to_string)

def column_review(df,col,cardinality_threshold = 100):

    nulls = (df[col].isna()).sum()
    pct_null = nulls/(len(df[col]))
    print(f'{col} has {pct_null:.1%} nulls ({nulls} out of {len(df[col]):,})')

    if pd.api.types.is_string_dtype(df[col]):
        string_column(df,col,cardinality_threshold)
    
    if pd.api.types.is_any_real_numeric_dtype(df[col]) or pd.api.types.is_float_dtype(df[col]):
        number_column(df,col)

    if pd.api.types.is_datetime64_any_dtype(df[col]):
        date_column(df,col)
        
            


def dataset_review(df,cardinality_threshold = 100):

    
    print(f'Totals Rows = {len(df)}')
    print(f'Totals Columns = {len(df.columns)}') 
    print(f'Columns = {list(df.columns)}')
    print(df.dtypes)

    print('\n\n === Example Row ===')
    print(df.iloc[-1])

    for column in df.columns:

        column_review(df,column)



In [169]:
dataset_review(results)

Totals Rows = 49477
Totals Columns = 9
Columns = ['date', 'home_team', 'away_team', 'home_score', 'away_score', 'tournament', 'city', 'country', 'neutral']
date          datetime64[us]
home_team                str
away_team                str
home_score           float64
away_score           float64
tournament               str
city                     str
country                  str
neutral                 bool
dtype: object


 === Example Row ===
date          2026-06-27 00:00:00
home_team                 Croatia
away_team                   Ghana
home_score                    NaN
away_score                    NaN
tournament         FIFA World Cup
city                 Philadelphia
country             United States
neutral                      True
Name: 49476, dtype: object
date has 0.0% nulls (0 out of 49,477)


=== Summary Statistic date===
Earliest Date == 1872-11-30 00:00:00
Latest Date == 2026-06-27 00:00:00
date
2012-02-29    66
2016-03-29    64
2025-06-10    62
2025-03-25    6

In [170]:
dataset_review(shootouts)

Totals Rows = 678
Totals Columns = 5
Columns = ['date', 'home_team', 'away_team', 'winner', 'first_shooter']
date             datetime64[us]
home_team                   str
away_team                   str
winner                      str
first_shooter               str
dtype: object


 === Example Row ===
date             2026-06-06 00:00:00
home_team                  Lithuania
away_team                     Latvia
winner                     Lithuania
first_shooter              Lithuania
Name: 677, dtype: object
date has 0.0% nulls (0 out of 678)


=== Summary Statistic date===
Earliest Date == 1967-08-22 00:00:00
Latest Date == 2026-06-06 00:00:00
date
2016-06-03    5
2024-03-26    5
1986-10-01    4
2018-06-09    4
2021-07-06    4
2026-03-30    4
2018-06-03    3
2020-10-08    3
1973-06-14    2
1973-07-26    2
1984-03-14    2
1984-12-16    2
1985-12-25    2
1986-06-21    2
1989-04-23    2
1990-12-17    2
1991-01-21    2
1995-07-17    2
1995-12-02    2
1996-06-22    2
1996-06-26    2
1996

In [171]:
dataset_review(goalscorers)

Totals Rows = 47690
Totals Columns = 8
Columns = ['date', 'home_team', 'away_team', 'team', 'scorer', 'minute', 'own_goal', 'penalty']
date         datetime64[us]
home_team               str
away_team               str
team                    str
scorer                  str
minute              float64
own_goal               bool
penalty                bool
dtype: object


 === Example Row ===
date            2026-06-18 00:00:00
home_team               Switzerland
away_team    Bosnia and Herzegovina
team         Bosnia and Herzegovina
scorer                 Ermin Mahmić
minute                         90.0
own_goal                      False
penalty                       False
Name: 47689, dtype: object
date has 0.0% nulls (0 out of 47,690)


=== Summary Statistic date===
Earliest Date == 1916-07-02 00:00:00
Latest Date == 2026-06-18 00:00:00
date
2011-10-11    145
2008-10-11    136
2008-09-06    135
2023-11-16    135
2004-09-08    132
2004-10-13    128
2011-09-02    125
2011-09-06    12

In [ ]:
len(results[results['home_score'].isna()])

not_played = results[results['home_score'].isna()]
played = results[~results['home_score'].isna()]

played = played.assign(shootout_id = played[['date','home_team','away_team']].apply(
    lambda row: '_'.join([str(each) for each in row]), axis = 1
))

shootouts = shootouts.assign(shootout_id = shootouts[['date','home_team','away_team']].apply(
    lambda row: '_'.join([str(each) for each in row]), axis = 1
))

shootouts_ids = list(shootouts['shootout_id'].unique())

shootout_dict = pd.Series(shootouts['winner'].values, index = shootouts['shootout_id']).to_dict()

matches_with_shootouts = played[played['shootout_id'].isin(shootouts_ids)]

matches_with_shootouts = list(matches_with_shootouts['shootout_id'].unique())

print(len(shootouts_ids),len(matches_with_shootouts))

print(set(shootouts_ids)-set(matches_with_shootouts))


played['shootout_winner'] = played['shootout_id'].map(shootout_dict)


678 677
{'2011-06-29 00:00:00_Saare County_Åland Islands'}


,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral,shootout_id,shootout_winner
7124,1967-08-22,India,Taiwan,1.0,1.0,Merdeka Tournament,Kuala Lumpur,Malaysia,True,1967-08-22 00:00:00_India_Taiwan,Taiwan
8634,1971-11-14,South Korea,Vietnam Republic,1.0,1.0,King's Cup,Bangkok,Thailand,True,1971-11-14 00:00:00_South Korea_Vietnam Republic,South Korea
8819,1972-05-07,South Korea,Iraq,0.0,0.0,AFC Asian Cup,Bangkok,Thailand,True,1972-05-07 00:00:00_South Korea_Iraq,Iraq
8838,1972-05-17,Thailand,South Korea,1.0,1.0,AFC Asian Cup,Bangkok,Thailand,False,1972-05-17 00:00:00_Thailand_South Korea,South Korea
8841,1972-05-19,Thailand,Cambodia,2.0,2.0,AFC Asian Cup,Bangkok,Thailand,False,1972-05-19 00:00:00_Thailand_Cambodia,Thailand
...,...,...,...,...,...,...,...,...,...,...,...
49212,2026-03-30,Gabon,Trinidad and Tobago,2.0,2.0,FIFA Series,Tashkent,Uzbekistan,True,2026-03-30 00:00:00_Gabon_Trinidad and Tobago,Gabon
49213,2026-03-30,Uzbekistan,Venezuela,0.0,0.0,FIFA Series,Tashkent,Uzbekistan,False,2026-03-30 00:00:00_Uzbekistan_Venezuela,Uzbekistan
49252,2026-03-31,Bosnia and Herzegovina,Italy,1.0,1.0,FIFA World Cup qualification,Zenica,Bosnia and Herzegovina,False,2026-03-31 00:00:00_Bosnia and Herzegovina_Italy,Bosnia and Herzegovina
49255,2026-03-31,Czech Republic,Denmark,2.0,2.0,FIFA World Cup qualification,Prague,Czech Republic,False,2026-03-31 00:00:00_Czech Republic_Denmark,Czech Republic


In [212]:
played['goal_diff'] = np.abs(played['home_score']-played['away_score'])
played['year'] = played['date'].dt.year
played_long = played.melt(
    id_vars=['date','year','tournament'],
    value_vars=['home_team','away_team'],
    value_name='team'
)


wc_matches = played_long[(played_long['tournament'] == 'FIFA World Cup') & (played_long['year'] == 2026)]
wc_teams = wc_matches['team'].unique()
wc_former_names = former_names[former_names['current'].isin(wc_teams)]

name_map = dict(zip(wc_former_names['former'], wc_former_names['current']))

played['home_team'] = played['home_team'].replace(name_map)
played['away_team'] = played['away_team'].replace(name_map)

match_conditions = [
    played['home_score'] > played['away_score'],
    played['home_score'] < played['away_score'],
    played['shootout_winner'] == played['home_team'],
    played['shootout_winner'] == played['away_team']
]  

choices = ['W','L','W','L']
played['result'] = np.select(match_conditions,choices,default='D')




In [232]:
current_champion = None
reign_log = []

for row in played.itertuples(index = True,name='Match'):
    
    if current_champion is not None and current_champion not in (row.home_team, row.away_team):
        continue
    
    if row.result == 'W':
        challenger = row.home_team
    elif row.result == 'L':
        challenger = row.away_team
    else:
        continue

    if challenger != current_champion:
        reign_log.append({
            'date':row.date,
            'new_champion':challenger,
            'previous_champion':current_champion
        })
        current_champion = challenger

reigns = pd.DataFrame(reign_log)
reigns['reign_start'] = pd.to_datetime(reigns['date'])
reigns['reign_end'] = reigns['reign_start'].shift(-1)
reigns['reign_end'] = reigns['reign_end'].fillna(played['date'].max())
reigns['days_reign'] = (reigns['reign_end'] - reigns['reign_start']).dt.days

reigns


,date,new_champion,previous_champion,reign_start,reign_end,days_reign
0,1873-03-08,England,NaN,1873-03-08,1874-03-07,364
1,1874-03-07,Scotland,England,1874-03-07,1879-04-05,1855
2,1879-04-05,England,Scotland,1879-04-05,1880-03-13,343
3,1880-03-13,Scotland,England,1880-03-13,1888-03-17,2926
4,1888-03-17,England,Scotland,1888-03-17,1889-04-13,392
5,1889-04-13,Scotland,England,1889-04-13,1891-04-04,721
6,1891-04-04,England,Scotland,1891-04-04,1896-04-04,1827
7,1896-04-04,Scotland,England,1896-04-04,1898-04-02,728
8,1898-04-02,England,Scotland,1898-04-02,1900-04-07,735
9,1900-04-07,Scotland,England,1900-04-07,1903-03-21,1078


In [234]:
played[played['date'] == dt.datetime(2005,11,16)]

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral,shootout_id,shootout_winner,goal_diff,year,result
29639,2005-11-16,Australia,Uruguay,1.0,0.0,FIFA World Cup qualification,Sydney,Australia,False,2005-11-16 00:00:00_Australia_Uruguay,Australia,1.0,2005,W
29640,2005-11-16,Bahrain,Trinidad and Tobago,0.0,1.0,FIFA World Cup qualification,Riffa,Bahrain,False,2005-11-16 00:00:00_Bahrain_Trinidad and Tobago,NaN,1.0,2005,L
29641,2005-11-16,Cyprus,Wales,1.0,0.0,Friendly,Limassol,Cyprus,False,2005-11-16 00:00:00_Cyprus_Wales,NaN,1.0,2005,W
29642,2005-11-16,Czech Republic,Norway,1.0,0.0,FIFA World Cup qualification,Prague,Czech Republic,False,2005-11-16 00:00:00_Czech Republic_Norway,NaN,1.0,2005,W
29643,2005-11-16,Egypt,Tunisia,1.0,2.0,Friendly,Cairo,Egypt,False,2005-11-16 00:00:00_Egypt_Tunisia,NaN,1.0,2005,L
29644,2005-11-16,Georgia,Jordan,3.0,2.0,Friendly,Tbilisi,Georgia,False,2005-11-16 00:00:00_Georgia_Jordan,NaN,1.0,2005,W
29645,2005-11-16,Greece,Hungary,2.0,1.0,Friendly,Piraeus,Greece,False,2005-11-16 00:00:00_Greece_Hungary,NaN,1.0,2005,W
29646,2005-11-16,Ivory Coast,Italy,1.0,1.0,Friendly,Geneva,Switzerland,True,2005-11-16 00:00:00_Ivory Coast_Italy,NaN,0.0,2005,D
29647,2005-11-16,Japan,Angola,1.0,0.0,Friendly,Tokyo,Japan,False,2005-11-16 00:00:00_Japan_Angola,NaN,1.0,2005,W
29648,2005-11-16,South Korea,Serbia,2.0,0.0,Friendly,Seoul,South Korea,False,2005-11-16 00:00:00_South Korea_Serbia,NaN,2.0,2005,W


In [ ]:
#def build_elo_trend_by_team(df,team,base = 1000,runway = 30):




,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral,year,goal_diff,result,shootout_id
34873,2011-06-29,Afghanistan,Palestine,0.0,2.0,FIFA World Cup qualification,Tursunzoda,Tajikistan,True,2011,2.0,L,2011-06-29 00:00:00_Afghanistan_Palestine
34874,2011-06-29,Bangladesh,Pakistan,3.0,0.0,FIFA World Cup qualification,Dhaka,Bangladesh,False,2011,3.0,W,2011-06-29 00:00:00_Bangladesh_Pakistan
34875,2011-06-29,Cambodia,Laos,4.0,2.0,FIFA World Cup qualification,Phnom Penh,Cambodia,False,2011,2.0,W,2011-06-29 00:00:00_Cambodia_Laos
34876,2011-06-29,Iraq,Syria,1.0,2.0,Friendly,Erbil,Iraq,False,2011,1.0,L,2011-06-29 00:00:00_Iraq_Syria
34877,2011-06-29,Malaysia,Taiwan,2.0,1.0,FIFA World Cup qualification,Kuala Lumpur,Malaysia,False,2011,1.0,W,2011-06-29 00:00:00_Malaysia_Taiwan
34878,2011-06-29,Mongolia,Myanmar,1.0,0.0,FIFA World Cup qualification,Ulaanbaatar,Mongolia,False,2011,1.0,W,2011-06-29 00:00:00_Mongolia_Myanmar
34879,2011-06-29,Nepal,Timor-Leste,2.0,1.0,FIFA World Cup qualification,Kathmandu,Nepal,False,2011,1.0,W,2011-06-29 00:00:00_Nepal_Timor-Leste
34880,2011-06-29,Sri Lanka,Philippines,1.0,1.0,FIFA World Cup qualification,Colombo,Sri Lanka,False,2011,0.0,D,2011-06-29 00:00:00_Sri Lanka_Philippines
34881,2011-06-29,Vietnam,Macau,6.0,0.0,FIFA World Cup qualification,Ho Chi Minh City,Vietnam,False,2011,6.0,W,2011-06-29 00:00:00_Vietnam_Macau
